In [ ]:
import json
import pandas as pd

with open("../data/trainer_state.json") as f:
    state = json.load(f)

train_rows = [e for e in state["log_history"] if "loss" in e and "eval_loss" not in e]
eval_rows  = [e for e in state["log_history"] if "eval_impact_acc" in e]

train_df = pd.DataFrame(train_rows)
train_df["Epoch"] = train_df["epoch"].apply(lambda x: int(x) + 1)
avg_loss = train_df.groupby("Epoch")["loss"].mean().reset_index()
avg_loss.columns = ["Epoch", "Training Loss"]

eval_df = pd.DataFrame(eval_rows)
eval_df["Epoch"] = eval_df["epoch"].astype(int)
eval_df = eval_df.rename(columns={
    "eval_loss":              "Validation Loss",
    "eval_impact_acc":        "Impact Acc",
    "eval_urgency_acc":       "Urgency Acc",
    "eval_resource_f1_micro": "Resource F1 Micro",
    "eval_resource_f1_macro": "Resource F1 Macro",
})

table = avg_loss.merge(
    eval_df[["Epoch", "Validation Loss", "Impact Acc", "Urgency Acc",
             "Resource F1 Micro", "Resource F1 Macro"]],
    on="Epoch"
)
table

In [ ]:
print(table.to_latex(index=False, float_format="%.4f", 
caption="BERT multi-task training metrics per epoch.", label="tab:bert_metrics"))